In [ ]:
from pathlib import Path

import polars as pl
import yaml

with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

In [ ]:
device_pings = pl.DataFrame()
device_period = pl.DataFrame()
device_days = pl.DataFrame()
for i in range(256):
    data = pl.read_parquet(
        f"/mnt/common-hdd/group-dirs/tkom-2026-project/partitioned/{i:02x}/pings.parquet"
    )
    dpi = data.group_by(pl.col.device_id).agg(
        pl.col.timestamp.count().alias("number_of_pings")
    )
    dpo = data.group_by(pl.col.device_id).agg(
        pl.col.timestamp.min().alias("first"),
        pl.col.timestamp.max().alias("last"),
        (
            (
                pl.col.timestamp.max().dt.date() - pl.col.timestamp.min().dt.date()
            ).dt.total_days()
            + 1
        ).alias("interval_length"),
    )
    dd = (
        data.with_columns(pl.col("timestamp").dt.date().alias("day"))
        .group_by(pl.col.device_id)
        .agg(pl.col.day.unique().count().alias("number_of_days_active"))
    )
    device_pings = pl.concat([device_pings, dpi])
    device_period = pl.concat([device_period, dpo])
    device_days = pl.concat([device_days, dd])
    del data
device_pings = device_pings.sort("device_id")
device_period = device_period.sort("device_id")
device_days = device_days.sort("device_id")

# runs 9m

In [3]:
path = f"{config['dirs']['output']}/{config['city']}/descriptives"
Path(path).mkdir(exist_ok=True, parents=True)

device_stats = device_pings.join(device_period, on="device_id").join(
    device_days, on="device_id"
)
device_stats.write_parquet(f"{path}/device_stats.parquet")
device_stats.write_csv(f"{path}/device_stats.csv")

In [8]:
device_stats

device_id,number_of_pings,first,last,interval_length,number_of_days_active
i64,u32,"datetime[μs, Europe/Budapest]","datetime[μs, Europe/Budapest]",i64,u32
1,4466,2024-10-02 02:37:09 CEST,2025-12-04 08:48:18 CET,429,35
2,1876,2025-06-24 05:52:42 CEST,2025-07-25 14:51:22 CEST,32,25
3,26012,2024-10-01 02:01:05 CEST,2025-03-14 00:54:21 CET,165,163
4,213,2025-05-24 20:46:00 CEST,2025-05-25 11:54:21 CEST,2,2
5,1,2025-11-02 12:18:00 CET,2025-11-02 12:18:00 CET,1,1
…,…,…,…,…,…
1652456,628,2023-12-10 01:20:24 CET,2024-02-17 10:17:35 CET,70,14
1652457,334,2025-10-19 10:39:34 CEST,2025-11-02 16:55:14 CET,15,10
1652458,487,2025-06-22 13:43:16 CEST,2025-07-12 15:30:00 CEST,21,11
